In [9]:
from pathlib import Path
import pandas as pd

In [13]:
DATA_PATH = Path("../data/combined_dataset_labeled.xlsx")

df = pd.read_excel(
    DATA_PATH,
    sheet_name="Research Dataset"
)

df.shape

(1055, 84)

In [14]:
df["Tissue_Grade_Source"].value_counts(dropna=False)

Tissue_Grade_Source
Eye-bank form    1037
NaN                18
Name: count, dtype: int64

In [15]:
pd.crosstab(
    df["Dataset_Part"],
    df["Tissue_Grade_Source"],
    dropna=False
)

Tissue_Grade_Source,Eye-bank form,NaN
Dataset_Part,,
Graded set (eye-bank forms),1037,18


In [16]:
df["Tissue_Grade"].value_counts(dropna=False)

Tissue_Grade
A+                                                                             543
A                                                                              311
B                                                                              167
NaN                                                                             19
A (release form; tissue sheet A+)                                                3
A (tissue sheet) / A+ (release form)                                             2
A (as read - verify)                                                             1
A+ / A (both circled on tissue sheet - verify)                                   1
B (release form A changed to B by hand; tissue sheet A)                          1
A+ (release; "-> B" handwritten)                                                 1
A+ (release form; handwritten mark may change to B - verify)                     1
A+ / A (both circled)                                                     

In [17]:
pd.crosstab(
    df["Tissue_Grade_Source"],
    df["Tissue_Grade"],
    dropna=False
)

Tissue_Grade,A,A (as read - verify),A (release form; tissue sheet A+),A (release form; tissue sheet circles A+),A (tissue sheet) / A+ (release form),A+,A+ (as read),A+ (release form; handwritten mark may change to B - verify),"A+ (release; ""-> B"" handwritten)",A+ / A (both circled on tissue sheet - verify),A+ / A (both circled),B,"B (printed A+ on release form, handwritten downgrade to B; tissue sheet A+)",B (release form A changed to B by hand; tissue sheet A),B (release grade A changed to B by hand),NaN
Tissue_Grade_Source,,,,,,,,,,,,,,,,
Eye-bank form,311,1,3,1,2,543,1,1,1,1,1,167,1,1,1,1
NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,18


In [18]:
df["Needs_Review"].value_counts(dropna=False)

Needs_Review
NaN    1055
Name: count, dtype: int64

In [19]:
df.loc[
    df["Needs_Review"].notna(),
    ["Tissue_ID", "Tissue_Grade", "Tissue_Grade_Source",
     "Needs_Review", "Review_Notes"]
].head(20)

,Tissue_ID,Tissue_Grade,Tissue_Grade_Source,Needs_Review,Review_Notes


In [20]:
grade_counts = (
    df["Tissue_Grade"]
    .value_counts(dropna=False)
    .reset_index()
)

grade_counts.columns = ["Tissue_Grade", "Count"]

grade_counts

,Tissue_Grade,Count
0,A+,543
1,A,311
2,B,167
3,NaN,19
4,A (release form; tissue sheet A+),3
5,A (tissue sheet) / A+ (release form),2
6,A (as read - verify),1
7,A+ / A (both circled on tissue sheet - verify),1
8,B (release form A changed to B by hand; tissue...,1
9,"A+ (release; ""-> B"" handwritten)",1


In [21]:
standard_grades = ["A+", "A", "B"]

non_standard = df[
    ~df["Tissue_Grade"].isin(standard_grades)
]

non_standard[
    ["Tissue_ID", "Tissue_Grade", "Tissue_Grade_Source"]
].sort_values("Tissue_Grade")

,Tissue_ID,Tissue_Grade,Tissue_Grade_Source
74,NEB-2020-10-12729 R,A (as read - verify),Eye-bank form
226,NEB-2024-05-16056 R,A (release form; tissue sheet A+),Eye-bank form
227,NEB-2024-05-16057 L,A (release form; tissue sheet A+),Eye-bank form
364,NEB-2024-08-16297 R,A (release form; tissue sheet A+),Eye-bank form
468,NEB-2025-09-17520 R,A (release form; tissue sheet circles A+),Eye-bank form
244,NEB-2024-05-16074 R,A (tissue sheet) / A+ (release form),Eye-bank form
319,NEB-2024-08-16252 L,A (tissue sheet) / A+ (release form),Eye-bank form
414,NEB-2025-09-17466 R,A+ (as read),Eye-bank form
321,NEB-2024-08-16254 L,A+ (release form; handwritten mark may change ...,Eye-bank form
302,NEB-2024-06-16133 L,"A+ (release; ""-> B"" handwritten)",Eye-bank form


In [22]:
non_standard[
    ["Tissue_ID", "Tissue_Grade", "Tissue_Grade_Source"]
].to_string(index=False)

'                   Tissue_ID                                                                Tissue_Grade Tissue_Grade_Source\n12346 R (year/month unknown)                                                                         NaN                 NaN\n         NEB-2020-09-12695 L                                                                         NaN                 NaN\n         NEB-2020-09-12697 L                                                                         NaN                 NaN\n         NEB-2020-09-12698 L                                                                         NaN                 NaN\n         NEB-2020-09-12699 R                                                                         NaN                 NaN\n         NEB-2020-09-12700 L                                                                         NaN                 NaN\n         NEB-2020-09-12701 R                                                                         NaN             

In [23]:
review_grades = {
    "12729": "A",
    "16056": "A",
    "16057": "A",
    "16060": "A",
    "16061": "B",
    "16074": "A+",
    "16133": "B",
    "16252": "A+",
    "16254": "B",
    "16297": "A",
    "16603": "A",
    "17466": "A",
    "17482": "B",
    "17495": "B",
    "17520": "A",
}

# Keep the original grade untouched
df["Tissue_Grade_Clean"] = df["Tissue_Grade"]

# Extract the numeric tissue ID from Tissue_ID
tissue_number = (
    df["Tissue_ID"]
    .astype(str)
    .str.extract(r"(\d{5})")[0]
)

# Apply specialist-reviewed grades
mask = tissue_number.isin(review_grades)

df.loc[mask, "Tissue_Grade_Clean"] = tissue_number[mask].map(review_grades)

In [24]:
df.loc[
    mask,
    ["Tissue_ID", "Tissue_Grade", "Tissue_Grade_Clean"]
].sort_values("Tissue_ID")

,Tissue_ID,Tissue_Grade,Tissue_Grade_Clean
74,NEB-2020-10-12729 R,A (as read - verify),A
226,NEB-2024-05-16056 R,A (release form; tissue sheet A+),A
227,NEB-2024-05-16057 L,A (release form; tissue sheet A+),A
230,NEB-2024-05-16060 R,A+ / A (both circled on tissue sheet - verify),A
231,NEB-2024-05-16061 L,B (release form A changed to B by hand; tissue...,B
244,NEB-2024-05-16074 R,A (tissue sheet) / A+ (release form),A+
302,NEB-2024-06-16133 L,"A+ (release; ""-> B"" handwritten)",B
319,NEB-2024-08-16252 L,A (tissue sheet) / A+ (release form),A+
321,NEB-2024-08-16254 L,A+ (release form; handwritten mark may change ...,B
364,NEB-2024-08-16297 R,A (release form; tissue sheet A+),A


In [25]:
df["Tissue_Grade_Clean"].value_counts(dropna=False)

Tissue_Grade_Clean
A+     545
A      319
B      172
NaN     19
Name: count, dtype: int64

In [26]:
missing_grades = df[
    df["Tissue_Grade_Clean"].isna()
]

missing_grades[
    ["Tissue_ID", "Tissue_Grade", "Tissue_Grade_Source"]
].to_string(index=False)

'                   Tissue_ID Tissue_Grade Tissue_Grade_Source\n12346 R (year/month unknown)          NaN                 NaN\n         NEB-2020-09-12695 L          NaN                 NaN\n         NEB-2020-09-12697 L          NaN                 NaN\n         NEB-2020-09-12698 L          NaN                 NaN\n         NEB-2020-09-12699 R          NaN                 NaN\n         NEB-2020-09-12700 L          NaN                 NaN\n         NEB-2020-09-12701 R          NaN                 NaN\n         NEB-2020-10-12761 R          NaN                 NaN\n         NEB-2020-10-12763 L          NaN                 NaN\n         NEB-2020-10-12772 L          NaN                 NaN\n         NEB-2022-04-13848 R          NaN                 NaN\n14302 L (year/month unknown)          NaN                 NaN\n         NEB-2024-06-16116 L          NaN                 NaN\n         NEB-2024-06-16117 R          NaN                 NaN\n         NEB-2024-06-16137 L          NaN             

In [29]:
# Final grading summary
grade_summary = (
    df["Tissue_Grade_Clean"]
    .value_counts(dropna=False)
    .rename_axis("Tissue_Grade_Clean")
    .reset_index(name="Count")
)

print(grade_summary)

  Tissue_Grade_Clean  Count
0                 A+    545
1                  A    319
2                  B    172
3                NaN     19


In [33]:
print(df["ECD_CD_cells_per_mm2"].describe())

count     956.000000
mean     2581.553347
std       468.981048
min         0.000000
25%      2336.000000
50%      2591.000000
75%      2857.000000
max      4484.000000
Name: ECD_CD_cells_per_mm2, dtype: float64


In [34]:
print((df["ECD_CD_cells_per_mm2"] == 0).sum())

4


In [35]:
print(df.loc[
    df["ECD_CD_cells_per_mm2"] == 0,
    [
        "Tissue_ID",
        "Tissue_Grade_Clean",
        "ECD_CD_cells_per_mm2",
        "CV",
        "HEX_percent",
        "SD",
        "Total_Cell_Count"
    ]
])

               Tissue_ID Tissue_Grade_Clean  ECD_CD_cells_per_mm2   CV  \
677  NEB-2023-03-14825 R                  B                   0.0  0.0   
818  NEB-2023-05-15117 R                  B                   0.0  0.0   
833  NEB-2023-06-15153 R                  B                   0.0  0.0   
890  NEB-2023-07-15267 R                  B                   0.0  0.0   

     HEX_percent   SD  Total_Cell_Count  
677          0.0  0.0               0.0  
818          0.0  0.0               0.0  
833          0.0  0.0               0.0  
890          0.0  0.0               0.0  


In [38]:
konan_measurement_cols = [
    "ECD_CD_cells_per_mm2",
    "CV",
    "HEX_percent",
    "SD",
    "Total_Cell_Count",
    "Total_Area_um2"
]

invalid_konan = df["ECD_CD_cells_per_mm2"] == 0

df.loc[invalid_konan, konan_measurement_cols] = pd.NA

print(invalid_konan.sum())

0


In [39]:
df.loc[
    invalid_konan,
    ["Tissue_ID", "Tissue_Grade_Clean"] + konan_measurement_cols
]

,Tissue_ID,Tissue_Grade_Clean,ECD_CD_cells_per_mm2,CV,HEX_percent,SD,Total_Cell_Count,Total_Area_um2


In [40]:
df.loc[
    df["Tissue_ID"].isin([
        "NEB-2023-03-14825 R",
        "NEB-2023-05-15117 R",
        "NEB-2023-06-15153 R",
        "NEB-2023-07-15267 R"
    ]),
    ["Tissue_ID", "Tissue_Grade_Clean"] + konan_measurement_cols
]

,Tissue_ID,Tissue_Grade_Clean,ECD_CD_cells_per_mm2,CV,HEX_percent,SD,Total_Cell_Count,Total_Area_um2
677,NEB-2023-03-14825 R,B,NaN,NaN,NaN,NaN,NaN,NaN
818,NEB-2023-05-15117 R,B,NaN,NaN,NaN,NaN,NaN,NaN
833,NEB-2023-06-15153 R,B,NaN,NaN,NaN,NaN,NaN,NaN
890,NEB-2023-07-15267 R,B,NaN,NaN,NaN,NaN,NaN,NaN


In [42]:
print((df[konan_measurement_cols] == 0).sum())


ECD_CD_cells_per_mm2    0
CV                      0
HEX_percent             0
SD                      0
Total_Cell_Count        0
Total_Area_um2          0
dtype: int64


In [45]:
leakage_candidates = [
    "Overall_Donor_Quality",
    "Transplant_Suitability",
    "Suitability_Reason",
    "Tissue_Outcome",
    "Recipient_Age",
    "Recipient_Eye",
    "Recipient_Indication",
    "Request_Indication",
    "Primary_Procedure",
    "Secondary_Procedures",
    "Surgery_Date",
    "Requested_Grade",
    "Planned_Procedure",
    "Grade_Request_Met",
    "Review_Notes",
    "Additional_Findings"
]

for col in leakage_candidates:
    print(f"\n--- {col} ---")
    print(df[col].dropna().value_counts().head(10))


--- Overall_Donor_Quality ---
Overall_Donor_Quality
Excellent                                                          137
Very good                                                           79
Good                                                                31
Very good (as read)                                                  4
Excellent (as read)                                                  2
Good (as read)                                                       2
Good (Very good struck)                                              2
Excellent / very good (mark spans both - verify)                     2
Very good (Good also marked - verify)                                1
Very good (stroke runs across Excellent and Very good - verify)      1
Name: count, dtype: int64

--- Transplant_Suitability ---
Transplant_Suitability
Suitable (released for surgery - printed release form)                                                       197
Suitable                                   

In [46]:
leakage_columns = [
    "Overall_Donor_Quality",
    "Transplant_Suitability",
    "Suitability_Reason",
    "Tissue_Outcome",
    "Recipient_Age",
    "Recipient_Eye",
    "Recipient_Indication",
    "Request_Indication",
    "Primary_Procedure",
    "Secondary_Procedures",
    "Surgery_Date",
    "Requested_Grade",
    "Planned_Procedure",
    "Grade_Request_Met",
    "Review_Notes"
]

len(leakage_columns)

15

In [48]:
grade_mentions = df["Additional_Findings"].dropna()[
    df["Additional_Findings"].dropna().str.contains(
        r"\bGrade\s*(A\+|A|B)\b",
        case=False,
        regex=True,
        na=False
    )
]

print(grade_mentions)

156    Release form: Grade A+, irregular cut; rim 2.5...
157    Release form: Grade A+, irregular cut; rim 2.5...
165    Release form: Grade A+, irregular cut, pterygi...
173    Release form: Grade A, irregular cut, mild ict...
175    Release form: Grade A, irregular cut, mild ict...
                             ...                        
574    Release form: Grade A+, irregular cut; rim 3.0...
576    Release form: Grade B, irregular cut; rim 3.00...
584    Release form: Grade A, irregular cut; rim 2.00...
585    Release form: Grade A+, irregular cut; rim 3.0...
589    Release form: Grade A+, irregular cut, eye las...
Name: Additional_Findings, Length: 255, dtype: str


C:\Users\pulit\AppData\Local\Temp\ipykernel_2836\1925329226.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["Additional_Findings"].dropna().str.contains(


In [49]:
leakage_columns = [
    "Overall_Donor_Quality",
    "Transplant_Suitability",
    "Suitability_Reason",
    "Tissue_Outcome",
    "Recipient_Age",
    "Recipient_Eye",
    "Recipient_Indication",
    "Request_Indication",
    "Primary_Procedure",
    "Secondary_Procedures",
    "Surgery_Date",
    "Requested_Grade",
    "Planned_Procedure",
    "Grade_Request_Met",
    "Review_Notes",
    "Additional_Findings"
]

In [50]:
len(leakage_columns)

16